In [0]:
# Databricks notebook source

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, IntegerType, DateType, BooleanType, StringType
from pyspark.sql.window import Window

def transformSales_order_header(SalesOrderHeader_df):

    windowSpec_rw = Window.partitionBy("SalesOrderID").orderBy("OrderDate")
    statList = [1,2,3,4,5,6]
    SalesOrderHeader_df = SalesOrderHeader_df.filter(F.col("status").isin(statList)).withColumn(
		"rw", F.row_number().over(windowSpec_rw))
    SalesOrderHeader_df = SalesOrderHeader_df.filter(F.col("rw") == 1).drop("rw")
    SalesOrderHeader_df = SalesOrderHeader_df.withColumn("processed_timestamp", F.current_timestamp())
    SalesOrderHeader_df = SalesOrderHeader_df.select(                                
       F.col("SalesOrderID").cast(IntegerType()).alias("SalesOrderID"),
       F.col("RevisionNumber").cast(IntegerType()).alias("RevisionNumber"),
       F.col("OrderDate").cast(DateType()).alias("OrderDate"),
       F.col("DueDate").cast(DateType()).alias("DueDate"),
       F.col("ShipDate").cast(DateType()).alias("ShipDate"),
       F.col("Status").cast(IntegerType()).alias("Status"),
       F.col("OnlineOrderFlag").cast(BooleanType()).alias("OnlineOrderFlag"),
       F.trim(F.col("SalesOrderNumber").cast(StringType())).alias("SalesOrderNumber"),
       F.trim(F.col("PurchaseOrderNumber").cast(StringType())).alias("PurchaseOrderNumber"),
       F.col("AccountNumber").cast(StringType()).alias("AccountNumber"),
       F.col("CustomerID").cast(IntegerType()).alias("CustomerID"),
       F.col("SalesPersonID").cast(IntegerType()).alias("SalesPersonID"),
       F.col("TerritoryID").cast(IntegerType()).alias("TerritoryID"),
       F.col("BillToAddressID").cast(IntegerType()).alias("BillToAddressID"),
       F.col("ShipToAddressID").cast(IntegerType()).alias("ShipToAddressID"),
       F.col("ShipMethodID").cast(IntegerType()).alias("ShipMethodID"),
       F.col("CreditCardID").cast(IntegerType()).alias("CreditCardID"),
       F.col("CreditCardApprovalCode").cast(StringType()).alias("CreditCardApprovalCode"),
       F.col("CurrencyRateID").cast(IntegerType()).alias("CurrencyRateID"),
       F.col("SubTotal").cast(DecimalType(11, 0)).alias("SubTotal"),
       F.col("TaxAmt").cast(DecimalType(11, 0)).alias("TaxAmt"),
       F.col("Freight").cast(DecimalType(11, 0)).alias("Freight"),
       F.col("TotalDue").cast(DecimalType(11, 0)).alias("TotalDue"),
       F.col("Comment").cast(StringType()).alias("Comment"),
       F.col("rowguid").cast(StringType()).alias("rowguid"),
       F.col("ModifiedDate").cast(DateType()).alias("ModifiedDate"),
       F.col("_rescued_data").cast(StringType()).alias("_rescued_data"),
       F.col("processed_timestamp")
    )
                                 
    return SalesOrderHeader_df




if __name__ == "__main__":

    SalesOrderHeader_tbl = dbutils.widgets.get("SalesOrderHeader")
    SalesOrderHeader_df = df = spark.read.table(SalesOrderHeader_tbl)
    SalesOrderHeader_df_tgt = transformSales_order_header(SalesOrderHeader_df)
    display(SalesOrderHeader_df_tgt)